### Randomized Linear Algebra (Sketch-and-Project)
To solve Ax = B, if A is a square and invertible, we can just x = A^-1 B
But, it is not efficient when A is large. It also does not work when A is not invertible and/or not a square.
We can use direct solvers like Gaussian elimination, but they become computationally infeasible when A is too large.
We look at 2 methods here that optimizes the solving with iterative linear solvers.

1. Randomized Block Coordinate Descent (in numpy)
- Idea: Iteratively updates x by fixing few variables at a time. 
2. Randomized Block Kaczmarz (in JAX)
- Idea: Iteratively updates x by satisfying a few equations at a time.

In [ ]:
"""
Randomized Block Coordinate Descent

In test 1, we solve for x with A and b. 
1. Get 3 matrices. A_BB, A_B and b_B.
2. Compute residual with r_B = A_B @ x - b_B
3. Solve the smaller system A_BB @ d_B = r_B to find correction vector d_B
4. Minus correction vector d_B from guessed x
5. Iterate for num_iterations

In test 2, we create x artifically, then, we create b with a artifically generated symmetric positive definite matrix A.
This allows us to know the exact answer x to compare after we solved for x.

For symmetric positive definite matrix A, we use A = M^T M + λI. A is known as well-conditioned matrix.

https://ocw.mit.edu/courses/18-06sc-linear-algebra-fall-2011/804ab1e53134741d2044d241b50a285e_MIT18_06SCF11_Ses3.1sum.pdf

Some properities of symmetric positive definite matrices.
1. Symmetric: Perfect mirror image of itself across main diagonal (A = A^T). (symmetric matrices always have real eigenvalues)
2. All eigenvalues of A are strictly positive. (definition of positive definite)
3. Strictly Convex: For any non-zero vector v, the v^T A v is always strictly greater than zero. (definition of positive definite)
   This guarantees that the landscape has exactly one unique global minimum and zero flat spots.
   Proof:
   $f(\vec{v}) = \vec{v}^T A \vec{v} = \vec{v}^T \cdot (A\vec{v}) = C$ (where $C$ is a scalar).
   If $\vec{v}$ is an eigenvector, then $A\vec{v} = \lambda\vec{v}$. 
   Substituting this in gives $\vec{v}^T \cdot (\lambda\vec{v}) = \lambda(\vec{v}^T \cdot \vec{v})$.
   The dot product $\vec{v}^T \cdot \vec{v}$ for a 3D vector $\begin{bmatrix} a & b & c \end{bmatrix}^T$ is just $a^2 + b^2 + c^2$, 
   which is strictly a positive number as long as not all of $a, b, c = 0$. Therefore, if $\lambda > 0$, the resulting scalar $C$ is positive.
  
Point 2 and 3 are equivalent

We used A = M.T @ M + λ * Identity_nxn # We use λ = 10 here to improve conditioning, using λ = 0.1 the matrix takes very long to converge.
We are adding exactly scalar λ to every single eigenvalue of the matrix A, to increase the value of A's eigenvalues.

Geometric interpretation
When we set f(v) = v^T A v = C, C represents a level surface (e.g., a 3D surface if parameterized by a, b, c).
At the origin: f(\vec{0}) = 0. Expanding Outward: Increasing $C$ means the level surface moves further from the origin in $a, b, c$ space.
We shouldn't draw wobbly, irregular level surfaces for a symmetric matrix A. Because A is symmetric, it has perfectly orthogonal eigenvectors. 
This guarantees the level surfaces are perfectly symmetrical ellipsoids (or ellipses in 2D). 
The orthogonal eigenvectors of A form the exact axes of these ellipsoids.
Also, the eigenvalues tells us the steepness of the surface. Along these eigenvector axes, the function simplifies to λ_1 a^2 + λ_2 b^2 + λ_3 c^2 = C$.
If a specific eigenvalue is big, the function increases rapidly in that direction. 
This makes that part of the level surface very steep (the ellipsoid is squeezed tightly along that specific axis).

Optimization landscape
For any symmetric positive definite matrix A, solving Ax = b is equivalent to finding the vector x that minimizes the following quadratic function:
f(x) = 0.5x^T A x - b^T x
\nabla f(x) = Ax - b
To find the minimum of the landscape, we set the gradient to zero: Ax - b = 0 ==> Ax = b

Results
For small matrix A, guassian elimination is faster.
But for bigger matrix A, iterative solver is faster. However, it is sensitive to matrix conditioning, 
and run out of time before reaching minima, leading to inaccurate results if num_iterations is off.
"""

import numpy as np
import time

def sketch_and_project(A, b, num_iterations, block_size):
    n, m = A.shape # n is the number of rows of A
    # Set the initial guess x0 = 0
    x = np.zeros(m)
    for t in range(num_iterations):
        # Randomly choose a subset B of indices to pick out rows and columns
        B = np.random.choice(n, size=block_size, replace = False) # choose blocksize number of rows, do not pick same row idx twice
        # Now we need to extract the relevant submatrix and subvectors
        A_BB = A[np.ix_(B,B)] # This is matrix A with rows B and columns B only
        A_B = A[B,:] # This is matrix A with rows B but all columns. B,: means [b rows, all columns]
        b_B = b[B] # This is the vector b but with only rows B
        # Now we compute the *residual* of the equations
        r_B = A_B @ x - b_B
        # Solve the small linear system to get the correction vector d
        d_B = np.linalg.solve(A_BB, r_B)
        # Update the parts of the vector x in B (we minus the residual. if we computed the error instead, we should add the error)
        x[B] -= d_B
    return x

# Test 1: On a small matrix
A = np.array([
    [2.0, 1.0, 0.0],
    [1.0, 2.0, 1.0],
    [0.0, 1.0, 2.0]
])

b = np.array([1.0, 2.0, 3.0])

x_approx = sketch_and_project(
    A, b,
    num_iterations=500,
    block_size=2
)

print("Sketch-and-project solution 1:")
print(x_approx)

#Test 2: On a large matrix
np.random.seed(0) # for reproducibility

n = 5000 # size of the system
block_size = 20
num_iterations = 2000

# Step 1: true solution
x_true = np.random.randn(n)

# Step 2: construct a symmetric positive definite matrix A = M^T M + λI
M = np.random.randn(n, n)
A = M.T @ M + 10 * np.eye(n) # We use λ = 10 here to improve conditioning, using λ = 0.1 the matrix takes very long to converge

# Step 3: RHS
b = A @ x_true

start = time.perf_counter()

x_approx = sketch_and_project(
    A, b,
    num_iterations=num_iterations,
    block_size=block_size
)

end = time.perf_counter()
SAP_time = end - start

start_1 = time.perf_counter()

x_direct = np.linalg.solve(A, b)

end_1 = time.perf_counter()

GE_time = end_1 - start_1

solution_error = np.linalg.norm(x_approx - x_true)
print("Solution error:", solution_error)
rel_error = np.linalg.norm(x_approx - x_true) / np.linalg.norm(x_true)
print("Relative error:", rel_error)
print("Sketch-and-project time (seconds):", SAP_time)
print("Gaussian elimination time (seconds):", GE_time)

# Running the code above, we can solve the simple small matrix, getting the solution x = [5, 0, 1.5].
# Then solving a large system with a large matrix A, with n=1000, num_iterations=10000, block size = 50 we get good agreement between actual solution and the SAP solution. The results are:

# Solution error: 0.025848151018749183
# Relative error: 0.0008272593836802964
# Sketch-and-project time (seconds): 2.2151416929991683
# Gaussian elimination time (seconds): 0.016270128995529376

# But in such a case, we note that SAP is actually slower than GE! This is due to the large number of iterations, large block size, and also n is not large enough for SAP to truly shine.
# So this is where we see the flaws of SAP - it depends on well conditioned matrices that can convege fast enough.
# We run the code again with n=5000, num_iterations=2000, and block_size=20. Doing so, we get the following results:

# Solution error: 45.303167214777
# Relative error: 0.6501451919055787
# Sketch-and-project time (seconds): 0.2626515379961347
# Gaussian elimination time (seconds): 1.218804659001762

# Note that the solution error is larger in this case. So it will be an issue of balancing accuracy with speed, and well-conditioned matrices will reduce the need for big tradeoffs.

In [ ]:
"""
Randomized Block Kaczmarz

For standard Kaczmarz (block size = 1), we move our current guess x0 towards the intersection iteratively by:
1. Projecting the current guess orthogonally onto one hyperplane.
2. Projecting that new point orthogonally onto the next hyperplane.
3. Repeating for num_iterations until converging at the intersection.

Proof (for block size = 1):
For a single linear equation, we can write as a^T x = b, and a^T would be the normal vector of the hyperplane, x is the unknown vector, and b is the scalar target.
To move x0 perpendicularly down onto the plane, we just move it in the direction of the normal vector a.
To know how much in the direction to move, we solve x_new = x0 + αa and since x_new is on the plane, a^T (x_new) = b ==> a^T (x0 + αa) = b
Solving for α, we get α = \frac{b - a^T x_0}{a^T a} ==> x_{new} = x_0 + \frac{b - a^T x_0}{||a||^2} * a
a^T a is just the squared length of the normal vector a and a^T x_0 - b is the residual r.
Notice that the weight = \frac{-r}{||a||^2}. This is because our point is initially away from the hyperplane. We need to move in the negative of the normal vector, hence step size α must be negative.

By iteratively projecting the current guess onto the next hyperplanes, we converge to the answer (the intersections of all hyperplanes)
Proof: 
Let the current guess be x_k, the projected guess be x_{k+1} and the answer be x*
With Pythagorean identity, we have ||x_{k+1} - x*||^2 + ||x_{k+1} - x_k||^2 = ||x_k - x*||^2 ==> ||x_{k+1} - x*||^2 = ||x_k - x*||^2 - ||x_{k+1} - x_k||^2 
We see that the new squared error ||x_{k+1} - x*||^2 is equal to the old squared error, minus a positive number.
Hence, if step size (||x_{k+1} - x_k||^2) is non zero, the distance to the true solution strictly decreases with every single iteration.

For Block Kaczmarz (block size > 1), we project our current guess x0 directly onto the intersection of all the hyperplanes in our randomly selected block simultaneously.
Here, we cannot just follow one normal vector. We must move in a direction that is a linear combination of all the normal vectors in our block.
To do so, we use a Gram System. The projection uses the Gram Matrix G. Actually, for standard Kaczmarz, it is a special case where the block has a single row, so G is a 1x1 scalar a.

Gram Matrix G: A_B @ A_B.T
1. G is always symmetric (and square), hence invertible.
2. G is Positive Semi-Definite. Its eigenvalues are always non-negative λ >= 0.

From standard Kaczmarz, we have x_{new} = x_0 + \frac{b - a^T x_0}{||a||^2} * a ==> x_{new} = x_0 - \frac{r}{||a||^2} * a, where \frac{r}{||a||^2} is the weight of the projection for the current guess.
The weight is z in the Gram system. But, how do we derive the Gram System?

Derivation: 
A_B are our normal vectors, our movement direction is A_B^T z.
We start with x_{new} = x_0 - A_B^T z. Since x_{new} is on the intersection, A_B (x_{new}) = b_B ==> A_B (x_0 - A_B^T z) = b_B ==> A_B x_0 - b_B = (A_B A_B^T) z
Since residual r = A_B x_0 - b_B and A_B A_B^T is the Gram Matrix G, we have the system Gz = r

Gram System: (A_B @ A_B.T) z = r
We add a tiny jitter (small constant) to diagonal of G for numerical stability (ensuring λ > small constant) as some of the eigenvalues of G may be extremely close to 0.
Kaczmarz method treats each equation as a m-1 dimensional hyperplane in m dimensional space, where m is the total number of unknown variables (the no. columns of A_B).
The solution to the subset of linear equations is the intersection of the m-1 hyperplanes as mentioned above.
"""

import jax
import jax.numpy as jnp
from jax import lax, random, jit
import numpy as np # Used only for data generation
import time
from functools import partial

# 1. Enable 64-bit precision (Crucial for linear solvers)
jax.config.update("jax_enable_x64", True)

@partial(jit, static_argnames=['block_size', 'num_iterations'])
def sketch_and_project_jax(A, b, key, num_iterations, block_size):
    """
    Solves Ax=b using Randomized Block Kaczmarz with JAX JIT compilation.
    """
    n, m = A.shape
    x0 = jnp.zeros(m)

    # We define the body of the loop to be compiled
    def body_fun(i, val):
        x, current_key = val

        # Split key for randomness
        current_key, subkey = random.split(current_key)

        # 1. Randomly sample indices (rows)
        # Note: replace=False is computationally heavier in JAX than replace=True.
        # For very large N, replace=True is fine, but we stick to False for correctness.
        idx = random.choice(subkey, n, shape=(block_size,), replace=False)

        # 2. Extract Submatrices (Slicing)
        A_B = A[idx, :]
        b_B = b[idx]

        # 3. Compute Residual
        r = A_B @ x - b_B

        # 4. Solve the Gram system: (A_B @ A_B.T) z = r
        # We add a tiny jitter to diagonal for numerical stability
        gram = A_B @ A_B.T
        # gram = gram + 1e-10 * jnp.eye(block_size) # Uncomment if system is very ill-conditioned

        z = jnp.linalg.solve(gram, r)

        # 5. Update x
        x_new = x - A_B.T @ z

        return (x_new, current_key)

    # Run the loop efficiently using XLA
    # val is the tuple (initial_x, initial_key)
    final_x, _ = lax.fori_loop(0, num_iterations, body_fun, (x0, key))

    return final_x

# --- Test Setup ---

def run_benchmark():
    print("Generating data...")
    np.random.seed(0)

    # Size Parameters
    n = 35000
    block_size = 500
    num_iterations = 5000

    # Generate Data (using NumPy for setup)
    x_true = np.random.randn(n)
    M = np.random.randn(n, n)
    # Make A Symmetric Positive Definite (easier to solve)
    A_np = M.T @ M + 10 * np.eye(n)
    b_np = A_np @ x_true

    # Convert to JAX Arrays (device transfer)
    A_jax = jnp.array(A_np)
    b_jax = jnp.array(b_np)
    key = random.PRNGKey(42)

    print(f"System size: {n}x{n}")
    print(f"Iterations: {num_iterations}, Block size: {block_size}")
    print("-" * 30)

    # --- 1. JAX Sketch and Project ---

    print("Compiling JAX Kaczmarz...")
    # Trigger JIT compilation with a warm-up run (not timed)
    _ = sketch_and_project_jax(A_jax, b_jax, key, 10, block_size).block_until_ready()
    print("Compilation done. Running benchmark...")

    start_sap = time.perf_counter()

    # Actual Run
    x_approx = sketch_and_project_jax(A_jax, b_jax, key, num_iterations, block_size)
    # Important: JAX is asynchronous. We must block until result is ready to time it correctly.
    x_approx.block_until_ready()

    end_sap = time.perf_counter()
    time_sap = end_sap - start_sap

    # --- 2. JAX Direct Solve (Gaussian Elimination/LU) ---

    print("Running JAX Direct Solve (jnp.linalg.solve)...")
    start_direct = time.perf_counter()
    x_direct = jnp.linalg.solve(A_jax, b_jax)
    x_direct.block_until_ready()
    end_direct = time.perf_counter()
    time_direct = end_direct - start_direct

    # --- Results ---

    # Move results back to CPU for printing
    err = jnp.linalg.norm(x_approx - jnp.array(x_true))
    rel_err = err / jnp.linalg.norm(jnp.array(x_true))

    print("-" * 30)
    print(f"Sketch-and-Project Time:  {time_sap:.5f} s")
    print(f"Direct Solve Time:        {time_direct:.5f} s")
    print(f"SAP Relative Error:       {rel_err:.6f}")

    if time_sap < time_direct:
        print("\nResult: SAP was FASTER than Direct Solve.")
    else:
        print("\nResult: SAP was SLOWER than Direct Solve.")

if __name__ == "__main__":
    run_benchmark()

In [ ]:
"""
Symmetric matrix always have orthogonal eigenvectors.
Spectral Theorem -  For any real symmetric matrix, the eigenvectors corresponding to distinct eigenvalues are always orthogonal (v1 dot v2 = 0).  
If there are repeated eigenvalues, it is still possible to choose a set of orthogonal eigenvectors. (we can choose another set of eigenvectors in the eigenspace span by the initial set of eigenvectors that calculated from the repeated eigenvalues with Gram-Schmidt process.)

Spectral Decomposition
Because we can always find a set of n orthonormal eigenvectors for an n x n symmetric matrix, we say that A is orthogonally diagonalizable.
A = Q Λ Q.T, where,
Q is an orthogonal matrix where columns are orthonormal eigenvectors. (orthogonal matrix is square matrix with all column vectors are orthogonal and orthonormal means they are unit vectors)
Λ is a diagonal matrix containing the eigenvalues
Q.T is the transpose of Q and also the inverse Q^-1

We can actually think of an orthogonal matrix Q as a rotation (det = 1), or a reflection (det = -1), or a combination of both.
Hence, Q.T = Q^-1 is just the reverse of the transformation by Q.
This means there always exist an orthogonal matrix that rotates the basis to align with the eigenvectors and the inverse of that matrix that transforms the eigenvectors to align with the basis.
This is not possible if the eigenvector were not orthogonal in the first place.
For Λ, we can imagine a stretching of the axis by the corresponding eigenvalues.

Hence, spectral decomposition essentially express a complicated transformation as a 3 step simple rotation Q.T > scaling Λ > rotation back Q.

Singular Value Decomposition
Given any matrix A, we can get a symmetric matrix with either A.T A or A A.T.
These symmetric matrices are positive semi-definite (eigenvalues are non negative).
Square root of those eigenvalues arranged in decending order are the singular values of A.

Hence, any matrix A, we can get A = U Σ V.T, where
the columns of U are the normalized eigenvectors of A A.T, and 
the columns of V are the normalized eigenvectors of A.T A, and
Σ is a diagonal matrix that contains the singular values of A A.T (or equivalently, A.T A).

Spectral Theorem for SVD: SVD allows us to write any matrix A as a sum of rank 1 matrices. Since SVD sorts singular values from largest to smallest,
the first rank-1 matrix σ_1 u_1 v_1^T captures the most important information in the entire matrix A.

Note: For symmetric positive definite matrices, i-th eigenvalue = i-th singular value
"""

In [ ]:
"""
Optimizations

We used linalg.solve() multiple times in this notebook. However, this function for direct and exact solving is inefficient for very big block sizes. 
We can solve linear systems like G z = r approximately with much faster speeds.

Application to Gaussian Processes:
A GP initially generates an infinite number of possible smooth curves (the Prior). When we observe actual data points, we restrict the GP to only keep the curves that pass near our observed data (the Posterior). 
Collapsing this infinite set of curves down gives us our final output: a mean prediction (our best guess curve) and an uncertainty band.
To calculate that mean prediction, we must construct and solve a linear system (K + σ^2 I) z = y
We have matrix A: K + σ^2 I, where K is the kernel covariance matrix built by measuring the similarity between real data points and σ^2 I is the observation noise. 
In our test code, we simulated this exact structure using A = M.T @ M + 10 * np.eye(n), where M.T @ M acts as a dummy positive semi-definite Kernel matrix, and 10*I acts as the observation noise.

To speed up solving process, we can use faster algorithms.

Speed of iterative solvers depend on κ = σ_1 / σ_n (ratio of largest singular value to smallest)
Sometimes, the data has few very big singular values (that capture most of the information) and many very small singular values.
Also, some equations contain more information and describe A more than others. Hence, if we sample rows with probabilities proportional to their squared length (the row normal vector ||a_i||^2), the algorithm converges much faster.
However, it could still be computationally expensive. 
We can use Randomized Hadamard Transform to magically uniformizes the leverage scores of the matrix, making the rows nearly equal important / provide similar information. Then we can random sample it and get fast convergence with low computation cost.
Another trick is to use Nesterov's Acceleration.
It adds momentum to the Kaczmarz update which makes it reach global minimum faster.
"""